# Notebook 03: Parameter Extraction

**Objective:** Run global optimization + local refinement to extract SPICE parameters from I-V data.

We cover:
1. Generate synthetic target data with known parameters
2. Build multi-curve objective function
3. Run DE (global) + TRF (local) two-stage extraction
4. Compare extracted vs true parameters
5. Visualize fitting quality

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt

from src.device.mosfet import MOSFETLevel3, MOSFETParamsLevel3
from src.device.curves import generate_iv_curves
from src.extraction.objective import (
    ExtractionObjective, CurveData, params_to_vector, vector_to_params
)
from src.extraction.optimizer import two_stage_extraction
from src.viz.plots import plot_extraction_comparison, plot_optimization_trace

## 1. Generate Target Data

We create a MOSFET with known parameters to use as "ground truth".

In [2]:
# Randomize target parameters
rng = np.random.RandomState(42)
p_true = MOSFETParamsLevel3(
    VTH0=rng.uniform(0.3, 0.7),
    U0=rng.uniform(200, 500),
    THETA=rng.uniform(0.02, 0.10),
    VSAT=rng.uniform(6e6, 12e6),
    ETA0=rng.uniform(0.03, 0.08),
)

model_true = MOSFETLevel3(p_true)
id_vg, id_vd = generate_iv_curves(model_true)

print("True parameters:")
for name in ['VTH0', 'U0', 'THETA', 'VSAT', 'ETA0']:
    print(f"  {name:10s} = {getattr(p_true, name):.6g}")

True parameters:
  VTH0       = 0.449816
  U0         = 485.214
  THETA      = 0.0785595
  VSAT       = 9.59195e+06
  ETA0       = 0.0378009


## 2. Build Extraction Objective

In [3]:
tg_vg = CurveData(vgs=id_vg.vgs, vds=id_vg.vds_values, ids=id_vg.ids)
tg_vd = CurveData(vgs=id_vd.vgs_values, vds=id_vd.vds, ids=id_vd.ids)

objective = ExtractionObjective(
    target_idvg=tg_vg,
    target_idvd=tg_vd,
    weights=(1.0, 0.3, 0.2),
)

print(f"Objective at true params: {objective(params_to_vector(p_true), MOSFETLevel3):.6e}")

Objective at true params: 0.000000e+00


## 3. Run Two-Stage Extraction

- **Stage 1:** Differential Evolution (50 iterations, global coarse search)
- **Stage 2:** TRF (local fine-tuning)

In [4]:
result = two_stage_extraction(
    objective,
    stage1="de",
    stage1_options={"max_iter": 80, "pop_size": 20},
    verbose=True,
)

Stage 1: Global search (DE) on 14 parameters
  Parameters: ['W', 'L', 'VTH0', 'GAMMA', 'PHI', 'U0', 'THETA', 'VSAT', 'KAPPA', 'LAMBDA', 'ETA0', 'N0', 'RD', 'RS']


C:\Users\11519\spice-model-toolkit\notebooks\..\src\device\mosfet.py:354: RuntimeWarning: divide by zero encountered in divide
  ratio = np.where(v_dsat > 1e-12, vds / v_dsat, 1e6)


C:\Users\11519\spice-model-toolkit\notebooks\..\src\device\mosfet.py:354: RuntimeWarning: invalid value encountered in divide
  ratio = np.where(v_dsat > 1e-12, vds / v_dsat, 1e6)


differential_evolution step 1: f(x)= 1.4263567837796145


differential_evolution step 2: f(x)= 1.4263567837796145


...

differential_evolution step 77: f(x)= 0.4829131601703197


differential_evolution step 78: f(x)= 0.4829131601703197


differential_evolution step 79: f(x)= 0.4829131601703197


differential_evolution step 80: f(x)= 0.4829131601703197
  Stage 1 complete: cost = 4.829132e-01

Stage 2: LM local refinement


  Stage 2 complete: cost = 1.925598e-01
  Total NFev: 23520

  Extracted parameters:
    W            = 0.000018
    L            = 2.5617e-07
    VTH0         = 0.449032
    GAMMA        = 1.2172
    PHI          = 0.459874
    U0           = 556.3109
    THETA        = 0.253639
    VSAT         = 6.7183e+06
    KAPPA        = 1.3412
    LAMBDA       = 0.078182
    ETA0         = 0.035697
    N0           = 1.3935
    RD           = 14.2479
    RS           = 19.6411


## 4. Compare Extracted vs True Parameters

In [5]:
x_ext = result["x_refined"]
param_names = result["param_names"]

print(f"{'Parameter':<12s} {'True':>12s} {'Extracted':>12s} {'Error%':>8s}")
print("-" * 50)
errors = []
for i, name in enumerate(param_names):
    if hasattr(p_true, name):
        true_val = getattr(p_true, name)
        ext_val = x_ext[i]
        err = abs(true_val - ext_val) / max(abs(true_val), 1e-12) * 100
        errors.append(err)
        print(f"{name:<12s} {true_val:>12.4g} {ext_val:>12.4g} {err:>7.2f}%")

n_good = sum(1 for e in errors if e < 5.0)
print(f"\nParameters within 5% error: {n_good}/{len(errors)}")
print(f"Mean error: {np.mean(errors):.2f}%")

Parameter            True    Extracted   Error%
--------------------------------------------------
W                   1e-05    1.759e-05   75.88%
L                 1.8e-07    2.562e-07   42.31%
VTH0               0.4498        0.449    0.17%
GAMMA                 0.4        1.217  204.31%
PHI                  0.65       0.4599   29.25%
U0                  485.2        556.3   14.65%
THETA             0.07856       0.2536  222.86%
VSAT            9.592e+06    6.718e+06   29.96%
KAPPA                 0.5        1.341  168.25%
LAMBDA               0.05      0.07818   56.36%
ETA0               0.0378       0.0357    5.57%
N0                    1.5        1.394    7.10%
RD                     20        14.25   28.76%
RS                     20        19.64    1.79%

Parameters within 5% error: 2/14
Mean error: 63.37%


## 5. Visualize Fit Quality

In [6]:
# Generate curves from extracted parameters
p_ext = vector_to_params(x_ext, MOSFETLevel3)
model_ext = MOSFETLevel3(p_ext)
ext_vg, ext_vd = generate_iv_curves(model_ext)

fig = plot_extraction_comparison(id_vg, ext_vg, vds_idx=-1,
                                 title="Target vs Extracted")
plt.show()

C:\Users\11519\AppData\Local\Temp\ipykernel_28684\1436238613.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Optimization Convergence

In [7]:
if "history" in result.get("stage1_info", {}):
    fig = plot_optimization_trace(result["stage1_info"]["history"])
    plt.show()
else:
    print("DE optimizer doesn't provide per-iteration history.")

DE optimizer doesn't provide per-iteration history.
